In [ ]:
import os, time, requests
from typing import List, Optional

# Load LlamaIndex 核心元件
from llama_index.core import StorageContext, Settings
from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.core.storage.docstore.types import BaseDocumentStore
from llama_index.core.indices.tree.base import TreeIndex
from llama_index.core.schema import Document
from llama_index.core import PromptTemplate 

# Load LlamaIndex 的 Ollama 整合
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

class LlamaIndexRAG:
    """
    使用 LlamaIndex 的 TreeIndex 實現 RAG 系統
    """
    def __init__(self, model_name="llama3:8b", embed_model_name="mxbai-embed-large", base_url="http://localhost:11434"):
        """
        初始化 RAG 系統
        Args:
            model_name: Ollama 模型名稱 (用於生成)
            embed_model_name: Ollama 嵌入模型名稱 (用於向量化，TreeIndex不直接使用但為LlamaIndex推薦配置)
            base_url: Ollama 服務 URL
        """
        self.model_name = model_name
        self.embed_model_name = embed_model_name
        self.base_url = base_url
        self.index = None         
        self._check_ollama_status()     # 檢查 Ollama 服務是否運行   
        # 配置 LlamaIndex 的 LLM 和嵌入模型
        self.llm = Ollama(model=self.model_name, base_url=self.base_url, request_timeout=30000)        
        self.embed_model = OllamaEmbedding(model_name=self.embed_model_name, base_url=self.base_url)        
        Settings.llm = self.llm                  # 全域配置 LlamaIndex 設定
        Settings.embed_model = self.embed_model
        Settings.chunk_size, Settings.chunk_overlap = 4096, 0
        # TreeIndex 使用的提示詞，可自訂
        self.qa_template = SimpleInputPrompt("""
請根據提供的參考資料回答用戶問題。

【參考資料】        
---------------------
{context_str}
---------------------
問題：{query_str}

【指令規範】
1. **全面提取**：若資料包含答案，請以繁體中文清楚、完整地列出不得遺漏。
2. **忠於原文**：保持客觀，絕對不能因總結而遺漏任何資訊，優先使用資料內的用語。
3. **誠實原則**：如果資料中確實完全沒有提到相關資訊，請禮貌地回答「根據目前的資料庫，無法提供相關資訊」。
4. **完整性**：忽略資料中不相關的雜訊（如：分頁標記、無意義的編號）。
5. **後驗檢查**：在輸出答案前，請再次確認是否遺漏了【參考資料】中任何符合問題條件的細節。
        """)
        
    def _check_ollama_status(self):
        """檢查 Ollama 服務器是否可達"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                print(f"✅ Ollama 服務器 {self.base_url} 可達。")
            else:
                raise ConnectionError(f"無法連接到 Ollama 服務器。狀態碼: {response.status_code}")
        except requests.exceptions.ConnectionError as e:
            raise ConnectionError(f"無法連接到 Ollama 服務器，請檢查服務是否已啟動: {e}")

    def create_index_from_documents(self, documents: List[str]):
        """
        從文檔列表建立 LlamaIndex TreeIndex        
        Args:
            documents: 包含所有頁面內容的字串列表
        """
        print("📝 正在載入並處理文檔...")
        
        start_time = time.time()     # 記錄開始時間

        # 將文檔列表轉換為 LlamaIndex 的 Document 物件
        docs = [Document(text=d) for d in documents]    
        # 建立 TreeIndex
        SUMMARY_PROMPT_TMPL = (
            "以下是部分文檔內容：\n"
            "{context_str}\n"
            "請使用繁體中文對上述內容進行簡明扼要的總結。"
        )
        summary_template = PromptTemplate(SUMMARY_PROMPT_TMPL)

        self.index = TreeIndex.from_documents(docs, summary_template=summary_template, insert_batch_size=1)
        
        end_time = time.time()            # 記錄結束時間        
        duration = end_time - start_time  # 計算並列印花費時間
        
        print(f"✅ 已成功建立 TreeIndex，總共 {len(docs)} 個文檔節點。")
        print(f"⏱️ 總共花費時間：{duration:.2f} 秒。")

    def _print_node_recursive(self, node_id: str, index_struct, docstore, depth: int, max_depth: int):
        """
        遞迴地列印節點及其子節點。
        """
        if depth > max_depth: return            
        indent = "  " * depth     
        try:
            node = docstore.get_document(node_id)
            if node is None:
                print(f"{indent}❌ 錯誤：節點 ID {node_id} 不存在於 docstore 中。")
                return
            summary = node.get_content()[:100].replace("\n", " ") + "..."
            print(f"{indent}└─ Node ID: {node.node_id} (Depth: {depth}) - Summary: {summary}")

            # 這是關鍵修改：從 index_struct 中獲取子節點 ID, 這樣可以確保我們只在有子節點關係的節點上進行遞迴
            child_node_ids = index_struct.get_children(node)
            if child_node_ids:
                for child_key, child_id in child_node_ids.items():
                    self._print_node_recursive(child_id, index_struct, docstore, depth + 1, max_depth)
        except Exception as e:
            print(f"{indent}❌ 錯誤：處理節點 {node_id} 時發生異常。原因: {e}")

    def print_tree_structure_recursive(self, max_depth: int = 3):
        """
        列印 TreeIndex 的樹狀架構，從根節點開始遞迴遍歷。        
        Args:
            max_depth: 樹狀結構的最大列印深度，避免輸出過長。
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
            
        print("\n🌳 TreeIndex 架構 (遞迴遍歷):")
        print("=" * 20)
        index_struct = self.index.index_struct
        docstore = self.index.docstore

        if not index_struct.root_nodes:
            print("❌ 錯誤：索引結構中沒有根節點。")
            return
        
        # 遍歷所有根節點並開始遞迴列印
        for root_key, root_id in index_struct.root_nodes.items():
            self._print_node_recursive(root_id, index_struct, docstore, depth=0, max_depth=max_depth)
        
        print("=" * 20)

    def print_tree_index_structure(self, max_depth: int = 2):
        """
        列印 TreeIndex 的樹狀架構。此方法會使用更穩健的方式來遍歷並列印節點。        
        Args:
            max_depth: 樹狀結構的最大列印深度，避免輸出過長。
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
        print("\n🌳 TreeIndex 架構:\n","=" * 20)
        
        all_nodes = list(self.index.docstore.docs.values())  # 直接從 docstore 中獲取所有節點。      
        if not all_nodes:
            print("❌ 錯誤：docstore 中沒有找到任何節點。")
            return
        print(f"共有 {len(all_nodes)} 個節點。\n", "列印所有節點的摘要:")        
       
        for node in all_nodes:                           # 遍歷並列印每個節點的摘要
            summary = node.get_content()[:250].replace("\n", " ") + "..."    
            # 檢查元資料是否有檔案名稱和頁碼，是否有父節點
            file_name = node.metadata.get('file_name', 'N/A')
            page_label = node.metadata.get('page_label', 'N/A')
            parent_node_id = node.relationships.get('PARENT', 'N/A')            
            print("-" * 50)
            print(f"Node ID: {node.node_id}")
            print(f"來源檔案: {file_name}, 頁碼: {page_label}")
            print(f"父節點ID: {parent_node_id}")
            print(f"摘要: {summary}")           
        print("=" * 20)

    def rag_query(self, query: str) -> str:
            """
            對索引進行查詢並獲取 RAG 回答，並顯示相關的來源節點文本。            
            Args:
                query: 查詢字串                
            Returns:
                模型的回答
            """
            if self.index is None: return "錯誤：索引尚未建立，無法進行查詢。"
            
            print("🔍 正在查詢...")            
            # 建立一個查詢引擎，並傳入自訂的提示詞
            query_engine = self.index.as_query_engine(text_qa_template=self.qa_template)
            
            try:
                response = query_engine.query(query)          # 執行查詢
                print(f"🤖 回答: {response.response}\n---\n") # 顯示最終回答
                print("📄 來源文件 (Source Nodes):")           # 顯示相關的來源節點文本
                if response.source_nodes:
                    for i, node in enumerate(response.source_nodes):
                        node_content = node.get_content().strip()
                        print(f"[{i+1}] Node ID: {node.node_id}")
                        print(f"內容摘要: {node_content[:200]}...") # 只顯示前200字，避免輸出過長
                        print("-" * 20)
                else:
                    print("❌ 找不到相關的來源節點。")
                return response.response                      # 回傳最終回答
            except Exception as e:
                return f"❌ 查詢時發生錯誤：{e}"
                
def main(data_file, model_name="llama3:8b", embed_model_name="mxbai-embed-large"):
    """
    使用示例
    """
    print(f"🤖  使用 LlamaIndex TreeIndex 的 {model_name}  RAG系統測試\n", "=" * 50)    
    try:
        rag = LlamaIndexRAG(model_name, embed_model_name)
    except ConnectionError as e:
        print(f"❌ 錯誤：{e}")
        return
        
    file_path = data_file
    if not os.path.exists(file_path):
        print(f"❌ 錯誤：找不到文件 '{file_path}'。請確認文件路徑是否正確。")
        return
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    documents = content.split("=== 第")
    documents = documents[1:]
    documents = ["=== 第" + page.strip() for page in documents]    
    print(f"共讀入 {len(documents)} 頁。")
    for idx in range(len(documents)):
      print(documents[idx][:20])
    print("= ="*20)
    
    rag.create_index_from_documents(documents)   
    rag.print_tree_index_structure()   # 列印 TreeIndex 架構
    print("="*50)
    rag.print_tree_structure_recursive(max_depth=3) # 您可以調整 max_depth
    
    print("\n🔍 請輸入查詢問題（輸入 quit 結束）\n", "=" * 60)    
    while True:
        query = input("❓ 問題: ").strip()
        
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
        
        answer = rag.rag_query(query)
        print(f"🤖 回答: {answer}\n","-" * 50)

if __name__ == "__main__":
    main("114_碩士班招生簡章_部分3.txt", 
         model_name="gpt-oss:20b",
         embed_model_name="bge-m3"
        )

🤖  使用 LlamaIndex TreeIndex 的 gpt-oss:20b  RAG系統測試
✅ Ollama 服務器 http://localhost:11434 可達。
共讀入 13 頁。
=== 第11 頁 ===
6 
 
系
=== 第12 頁 ===
7 
 
系
=== 第13 頁 ===
8 
系所代
=== 第14 頁 ===
9 
 
系
=== 第15 頁 ===
10 
  
=== 第16 頁 ===
11 
 

=== 第17 頁 ===
12 
  
=== 第18 頁 ===
13 
  
=== 第19 頁 ===
14 
 

=== 第20 頁 ===
15 
  
=== 第21 頁 ===
16 
  
=== 第22 頁 ===
17 
 

=== 第23 頁 ===
18 
 

= == == == == == == == == == == == == == == == == == == == =
📝 正在載入並處理文檔...
✅ 已成功建立 TreeIndex，總共 13 個文檔節點。
⏱️ 總共花費時間：545.77 秒。

🌳 TreeIndex 架構:
共有 15 個節點。
 列印所有節點的摘要:
--------------------------------------------------
Node ID: 657cb2d9-05a5-4e5e-825c-5487187d45e6
來源檔案: N/A, 頁碼: N/A
父節點ID: N/A
摘要: === 第11 頁 === 6    系所代碼  510  系所組別  新聞學系碩士班（一般生）  招生名額  4  報考資格  一、大學畢業具有學士學位者（含應屆）。  二、具同等學力資格者：（認定標準詳見本簡章附錄一）   (一)大學相關學系肄業持有修業證明書或休學證明書，並檢附歷年成績單。   (二)專科學校相關學科畢業者。     (三)高等考試或相當於高等考試之特種考試有關新聞行政人員、國際新聞及外交   人員及格，持有及格證書者。  系所指定 備審資料  書面審查一：  （一）自傳（...
--------------------------------------------------
Node ID: 36948201-e843-4

❓ 問題:  資訊管理學士碩士班 招生名額


🔍 正在查詢...
🤖 回答: 資訊管理學系碩士班（一般生） 的招生名額為 **9** 名。
---

📄 來源文件 (Source Nodes):
[1] Node ID: f575a620-bd77-42af-9bc3-13ee0c2bc823
內容摘要: === 第19 頁 ===
14 
 
系所代碼 
660 
系所組別 
資訊管理學系碩士班（一般生） 
招生名額 
9 
報考資格 
一、大學畢業具有學士學位者（含應屆）。 
二、具同等學力資格者：（認定標準詳見本簡章附錄一） 
(一)大學肄業持有修業證明書或休學證明書，並檢附歷年成績單。 
(二)專科學校畢業者。 
(三)高等考試或相當於高等考試之特種考試及格，持有及格證書者。 
系所指定
備...
--------------------
🤖 回答: 資訊管理學系碩士班（一般生） 的招生名額為 **9** 名。
 --------------------------------------------------


❓ 問題:  資管系招生名額多少


🔍 正在查詢...
